In [1]:
import pandas as pd 
df=pd.read_csv("cpcb_dly.csv")
df.head()

,Stn Code,Sampling Date,State,City/Town/Village/Area,Location of Monitoring Station,Agency,Type of Location,SO2,NO2,RSPM/PM10,PM 2.5
0,38,06-01-15,Tamil Nadu,Chennai,"Kathivakkam, Municipal Kalyana Mandapam, Chennai",Tamilnadu State Pollution Control Board,Industrial Area,11.0,14.0,50.0,NaN
1,38,20-01-15,Tamil Nadu,Chennai,"Kathivakkam, Municipal Kalyana Mandapam, Chennai",Tamilnadu State Pollution Control Board,Industrial Area,13.0,13.0,48.0,NaN
2,38,22-01-15,Tamil Nadu,Chennai,"Kathivakkam, Municipal Kalyana Mandapam, Chennai",Tamilnadu State Pollution Control Board,Industrial Area,11.0,15.0,42.0,NaN
3,38,27-01-15,Tamil Nadu,Chennai,"Kathivakkam, Municipal Kalyana Mandapam, Chennai",Tamilnadu State Pollution Control Board,Industrial Area,10.0,15.0,36.0,NaN
4,38,29-01-15,Tamil Nadu,Chennai,"Kathivakkam, Municipal Kalyana Mandapam, Chennai",Tamilnadu State Pollution Control Board,Industrial Area,13.0,12.0,35.0,NaN


In [2]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Dataset size:", df.size)

Number of rows: 2302
Number of columns: 11
Dataset size: 25322


In [3]:
print(df.columns.tolist())
print(df.dtypes)

['Stn Code', 'Sampling Date', 'State', 'City/Town/Village/Area', 'Location of Monitoring Station', 'Agency', 'Type of Location', 'SO2', 'NO2', 'RSPM/PM10', 'PM 2.5']
Stn Code                            int64
Sampling Date                         str
State                                 str
City/Town/Village/Area                str
Location of Monitoring Station        str
Agency                                str
Type of Location                      str
SO2                               float64
NO2                               float64
RSPM/PM10                         float64
PM 2.5                            float64
dtype: object


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2302 entries, 0 to 2301
Data columns (total 11 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Stn Code                        2302 non-null   int64  
 1   Sampling Date                   2302 non-null   str    
 2   State                           2302 non-null   str    
 3   City/Town/Village/Area          2302 non-null   str    
 4   Location of Monitoring Station  2302 non-null   str    
 5   Agency                          2302 non-null   str    
 6   Type of Location                2302 non-null   str    
 7   SO2                             2281 non-null   float64
 8   NO2                             2277 non-null   float64
 9   RSPM/PM10                       2295 non-null   float64
 10  PM 2.5                          454 non-null    float64
dtypes: float64(4), int64(1), str(6)
memory usage: 198.0 KB


In [6]:
df["Sampling Date"] = pd.to_datetime(
    df["Sampling Date"],
    format="%d-%m-%y",
    errors="coerce"
)

print("Start date:", df["Sampling Date"].min())
print("End date:", df["Sampling Date"].max())
print("Invalid dates:", df["Sampling Date"].isna().sum())

print("\nRecords per year:")
print(df["Sampling Date"].dt.year.value_counts().sort_index())

Start date: 2015-01-02 00:00:00
End date: 2015-12-31 00:00:00
Invalid dates: 0

Records per year:
Sampling Date
2015    2302
Name: count, dtype: int64


In [7]:
print("Exact duplicate rows:", df.duplicated().sum())

key_cols = [
    "Stn Code",
    "Sampling Date",
    "Location of Monitoring Station"
]

print(
    "Duplicate station-date combinations:",
    df.duplicated(subset=key_cols).sum()
)

Exact duplicate rows: 0
Duplicate station-date combinations: 0


In [8]:
df["Month"] = df["Sampling Date"].dt.month

print(
    df.groupby("Month")["PM 2.5"]
      .agg(
          total_records="size",
          available_values="count",
          average_pm25="mean"
      )
)

       total_records  available_values  average_pm25
Month                                               
1                201                 0           NaN
2                229                 0           NaN
3                244                34     37.076471
4                225                 0           NaN
5                198                59     30.650847
6                171                58     29.310345
7                199                56     29.410714
8                235                69     26.718841
9                 23                 0           NaN
10               221                63     31.666667
11               177                63     25.615873
12               179                52     29.759615


In [9]:
# Records per day in September
september = df[df["Sampling Date"].dt.month == 9]

print("September records:", len(september))
print("\nDates with records:")
print(september["Sampling Date"].value_counts().sort_index())

print("\nSeptember pollutant availability:")
print(september[["SO2", "NO2", "RSPM/PM10", "PM 2.5"]].notna().sum())

September records: 23

Dates with records:
Sampling Date
2015-09-01    3
2015-09-04    3
2015-09-08    3
2015-09-11    3
2015-09-15    3
2015-09-22    3
2015-09-26    3
2015-09-29    2
Name: count, dtype: int64

September pollutant availability:
SO2          23
NO2          23
RSPM/PM10    23
PM 2.5        0
dtype: int64


In [11]:
pollutant_cols = ["SO2", "NO2", "RSPM/PM10", "PM 2.5"]

print(df[pollutant_cols].describe().T)

# Count negative pollutant values
print("\nNegative values:")
print((df[pollutant_cols] < 0).sum())

            count       mean        std  min   25%   50%   75%    max
SO2        2281.0  10.656291   4.817755  2.0   7.0  11.0  14.0   40.0
NO2        2277.0  21.326307   6.775672  5.0  17.0  21.0  25.0   73.0
RSPM/PM10  2295.0  61.296296  33.088495  6.0  39.0  53.0  73.0  289.0
PM 2.5      454.0  29.550441  16.783704  4.0  18.0  25.0  36.0  141.0

Negative values:
SO2          0
NO2          0
RSPM/PM10    0
PM 2.5       0
dtype: int64


# Data Understanding Summary

## 1. Dataset Overview

The Tamil Nadu Air Quality dataset contains **2,302 records and 11 columns**, covering air quality observations collected during 2015. It includes monitoring station details, sampling dates, geographical information, monitoring agencies, location classifications, and four pollutant measurements.

## 2. Dataset Structure

The dataset contains three main categories of information:

- **Geographical and station information:** Station code, state, city/town/village/area, monitoring station location, and type of location.
- **Temporal information:** Sampling date, which enables monthly and seasonal analysis.
- **Pollutant measurements:** Sulphur dioxide (SO₂), nitrogen dioxide (NO₂), respirable suspended particulate matter/PM10 (RSPM/PM10), and fine particulate matter (PM2.5).

## 3. Key Observations

- The dataset covers a single calendar year, from 2 January 2015 to 31 December 2015.
- SO₂, NO₂ and RSPM/PM10 have relatively complete coverage, with less than 1.1% missing values in each column.
- PM2.5 has substantial missing data, with only 454 of 2,302 observations available (19.72%).
- September contains only 23 records, indicating uneven temporal coverage that should be considered during monthly comparisons.
- Initial validation identified no negative pollutant concentrations, exact duplicate rows, or invalid sampling dates.

## 4. Analytical Potential

The dataset can support analysis of monthly pollution patterns, comparisons between cities and monitoring stations, pollutant concentration distributions, and differences between monitoring location types. However, because it covers only one year, it cannot establish long-term annual trends or changes in current air quality.

## 5. Limitations

The main limitations are substantial missing PM2.5 measurements, uneven sampling coverage, and the single-year observation period. The measurement units and averaging periods should also be verified from the original data source before comparing pollutant concentrations with regulatory standards.

## Conclusion

The dataset provides a useful foundation for historical air quality analysis in Tamil Nadu. Its relatively complete SO₂, NO₂ and RSPM/PM10 measurements make it suitable for exploratory analysis and visualization, provided that missing data and sampling limitations are clearly documented.